<a href="https://colab.research.google.com/github/mj87246-ai/kfa-ai-agent/blob/main/2%EC%A3%BC%EC%B0%A8_%EC%82%AC%EC%A0%84%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KFA AI Agent Study — 2주차 사전과제 | LS ELECTRIC 원천 데이터 검증

**원본:** 교보증권 김광식, 「1Q26 Review: 떠오르는 글로벌 배전 시장의 신성」 (2026-04-22, 7쪽)  
**대상:** p.2 [도표 1] 실적 추이 및 전망 / p.3 [도표 3] 전력인프라 수주잔고 추이  
**목표:** PDF 원문 확인 → 산술 검산 → DART 실제 응답 조회 → 근거·한계 정리 → (선택) Gemini가 다음 확인 도구 결정.

**기존 SETUP.md 준수:** Colab Secrets에 `DART_API_KEY`, `GEMINI_API_KEY` 저장, 노트북 액세스 허용. 코드는 비밀 값을 출력하거나 저장하지 않음. 결과물은 `mj87246-ai/kfa-ai-agent`에 올림. [보고서 PDF 링크](https://stock.pstatic.net/stock-research/company/62/20260422_company_926341000.pdf) (PDF 파일 자체는 공개 GitHub에 업로드하지 않음).

> 이 노트북은 실시간 외부 API 실행 결과가 사전에 보장된 파일이 아닙니다. 셀 실행 후 실제 성공/실패 상태를 확인하세요. 리포트 작성 당시 정보와 나중에 발표된 확정 공시를 섞지 않습니다.

## 1. 원천 데이터 및 수집 방법

| 데이터 | 원천 후보 | 방식 | 확인해야 할 한계 |
|---|---|---|---|
| 연결 매출액·영업이익 | [OpenDART](https://opendart.fss.or.kr/), [재무정보 API 설명](https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS003) | ① 공식 API | 실제 호출은 개인 인증키 필요, 보고서의 P(잠정)와 최신 공시가 다를 수 있음 |
| 전력기기·전력인프라 실적 | [LS ELECTRIC IR](https://www.ls-electric.com/ko/company/invest/ir/) | ③ 공식 PDF/Excel 다운로드 | 연결/별도·사업부 집계 차이 확인 |
| 전력인프라 수주잔고 | [LS ELECTRIC IR](https://www.ls-electric.com/ko/company/invest/ir/) | ③ 자료 다운로드 및 수작업 대조 | p.3 그래프의 정확한 분기 값 및 오른쪽 축 정의는 원문 IR 재확인 필요 |
| 리포트 | [교보증권 리포트 링크](https://stock.pstatic.net/stock-research/company/62/20260422_company_926341000.pdf) | ③ PDF | 원문을 GitHub에 재배포하지 않음 |

*위 링크는 탐색 경로이며 현재 시점 API 응답 성공을 의미하지 않습니다.*

In [12]:
# Colab / Jupyter 공통 환경
import importlib.util, subprocess, sys
if importlib.util.find_spec('fitz') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pymupdf'])
import pymupdf as fitz
import re, io, zipfile, json
import pandas as pd
import requests
from pathlib import Path
print('환경 준비 완료')

환경 준비 완료


## 2. 리포트 PDF 로드 (사용자가 직접 업로드)

Google Colab에서는 실행 시 PDF 선택 창이 나타납니다. 로컬 Jupyter에서는 노트북과 같은 폴더에 `20260422_company_926341000.pdf`를 둡니다. **리포트 원문은 저장소에 올리지 않습니다.**

In [13]:
pdf_path = Path('20260422_company_926341000.pdf')
if not pdf_path.exists():
    try:
        from google.colab import files
        print('교보증권 PDF(20260422_company_926341000.pdf)를 업로드해 주세요.')
        uploaded = files.upload()
        if uploaded:
            pdf_path = Path(next(iter(uploaded.keys())))
    except ImportError:
        pass
if not pdf_path.exists():
    print('미확인: PDF가 없습니다. 다음 PDF 처리 셀은 실행하지 마세요.')
    report_text = None
else:
    with fitz.open(str(pdf_path)) as doc:
        print('PDF 페이지 수:', len(doc))
        report_text = '\n'.join(page.get_text(sort=True) for page in doc)
    print('확인:', pdf_path.name, '텍스트 추출 완료')

교보증권 PDF(20260422_company_926341000.pdf)를 업로드해 주세요.


Saving 교보증권_LS일렉트릭.pdf to 교보증권_LS일렉트릭 (1).pdf
PDF 페이지 수: 7
확인: 교보증권_LS일렉트릭 (1).pdf 텍스트 추출 완료


## 3. 리포트 내용 검증 — 수동 전사값과 PDF 대조

아래 숫자는 사용자가 제공한 **교보증권 2026-04-22 리포트 p.2 [도표 1]**에서 전사한 값입니다. 단위는 **십억원**. PDF 텍스트에서 관련 단어가 검출되는지도 확인하며, 글꼴·표 추출 방식 때문에 문자열 검색이 실패할 수 있으므로 **문자열 검출은 수치 독립 검증이 아닙니다**.

In [14]:
report_data = pd.DataFrame([
    {'구간':'1Q25','상태':'과거','매출액_십억원':1032.1,'영업이익_십억원':87.3},
    {'구간':'1Q26(P)','상태':'잠정','매출액_십억원':1376.6,'영업이익_십억원':126.6},
    {'구간':'2Q26(E)','상태':'추정','매출액_십억원':1536.7,'영업이익_십억원':156.7},
    {'구간':'FY25','상태':'잠정/기준확인','매출액_십억원':4962.2,'영업이익_십억원':426.9},
    {'구간':'FY26(E)','상태':'추정','매출액_십억원':6281.7,'영업이익_십억원':671.2},
])
report_data['영업이익률_계산(%)'] = (report_data['영업이익_십억원']/report_data['매출액_십억원']*100).round(2)
display(report_data)
if report_text is not None:
    for word in ['LS ELECTRIC 실적 추이 및 전망','전력인프라 수주잔고 추이']:
        print('PDF 제목 문자열 검출',word,':',word in report_text)

,구간,상태,매출액_십억원,영업이익_십억원,영업이익률_계산(%)
0,1Q25,과거,1032.1,87.3,8.46
1,1Q26(P),잠정,1376.6,126.6,9.20
2,2Q26(E),추정,1536.7,156.7,10.20
3,FY25,잠정/기준확인,4962.2,426.9,8.60
4,FY26(E),추정,6281.7,671.2,10.69


PDF 제목 문자열 검출 LS ELECTRIC 실적 추이 및 전망 : True
PDF 제목 문자열 검출 전력인프라 수주잔고 추이 : True


In [15]:
# 리포트 p.2의 1Q26 영업이익률(9.2%)과 1Q26 매출액 성장률(33.4%) 재계산
operating_margin = 126.6/1376.6*100
yoy_sales = (1376.6/1032.1-1)*100
print(f'1Q26 영업이익률 = {operating_margin:.3f}% (리포트 9.2%)')
print(f'1Q26 전년동기 매출 증가율 = {yoy_sales:.3f}% (리포트 p.3 33.4%)')
assert round(operating_margin,1)==9.2
assert round(yoy_sales,1)==33.4
print('계산 검산 통과 — 이는 보고서 내부 산술 검증이며 외부 공식 데이터 검증과는 다릅니다.')

1Q26 영업이익률 = 9.197% (리포트 9.2%)
1Q26 전년동기 매출 증가율 = 33.379% (리포트 p.3 33.4%)
계산 검산 통과 — 이는 보고서 내부 산술 검증이며 외부 공식 데이터 검증과는 다릅니다.


## 4. OpenDART: Colab Secrets 연결

SETUP.md에 지정된 비밀 이름은 **`DART_API_KEY`와 `GEMINI_API_KEY`**입니다. Colab 좌측 열쇠 메뉴에서 **노트북 액세스**를 켜야 합니다. 코드/출력/GitHub에 실제 키 값을 기재하지 않습니다.

2026년 4월 22일 발행 보고서의 `1Q26(P)`는 당시 잠정 정보입니다. 이후 발표된 DART 분기보고서는 **사후 교차검증**이며 보고서가 발표 당시 활용한 출처라고 주장할 수 없습니다.

In [16]:
# SETUP.md의 Colab 보안 비밀을 그대로 사용
from google.colab import userdata
DART_KEY = None
GEMINI_KEY = None
for label, name in [('DART', 'DART_API_KEY'), ('Gemini', 'GEMINI_API_KEY')]:
    try:
        secret = userdata.get(name)
        if label == 'DART':
            DART_KEY = secret
        else:
            GEMINI_KEY = secret
        print(f'{label}:', 'Secrets 연결됨' if secret else '키 없음')
    except Exception as exc:
        print(f'{label}: Secrets 접근 실패 ({type(exc).__name__}) — 노트북 액세스 확인')
# API 키 자체는 절대로 출력하지 않습니다.

DART: Secrets 연결됨
Gemini: Secrets 연결됨


In [17]:
def dart_get(endpoint, params):
    response = requests.get('https://opendart.fss.or.kr/api/'+endpoint,params=params,timeout=25)
    response.raise_for_status()
    result=response.json()
    if result.get('status')!='000':
        raise RuntimeError(f"DART 응답 {result.get('status')}: {result.get('message')}")
    return result

def lookup_corp_code(api_key, stock_code='010120'):
    r=requests.get('https://opendart.fss.or.kr/api/corpCode.xml',params={'crtfc_key':api_key},timeout=40)
    r.raise_for_status()
    import xml.etree.ElementTree as ET
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        root=ET.fromstring(z.read('CORPCODE.xml'))
    matches=[item for item in root.findall('list') if (item.findtext('stock_code') or '').strip()==stock_code]
    if not matches: raise ValueError('종목코드에 대응하는 DART 고유번호 미발견')
    return matches[0].findtext('corp_code')

def fetch_financials(api_key,corp_code,year='2026',reprt_code='11013'):
    return dart_get('fnlttSinglAcnt.json',{'crtfc_key':api_key,'corp_code':corp_code,'bsns_year':year,'reprt_code':reprt_code})

# 조회 성공 여부와 원본 응답을 별도로 남긴다.
dart_status='미확인'; dart_df=pd.DataFrame()
if DART_KEY:
    try:
        corp_code=lookup_corp_code(DART_KEY)
        print('LS ELECTRIC DART 고유번호 조회 완료')
        raw=fetch_financials(DART_KEY,corp_code)
        dart_df=pd.DataFrame(raw.get('list',[]))
        dart_status='공식 API 응답 수신'
        print('2026년 1분기 DART 응답:',len(dart_df),'개 항목')
        preview_cols=[x for x in ['fs_div','account_nm','thstrm_nm','thstrm_amount','currency'] if x in dart_df.columns]
        display(dart_df[preview_cols].head(20))
    except Exception as e:
        dart_status='조회 실패/검증 불가'
        print('DART 조회 실패:',type(e).__name__, '— 인증/서버/응답/네트워크 상태 확인 (인증 정보를 포함한 오류 원문은 출력하지 않음)')
else:
    print('DART API 생략 — 실제 조회를 수행하지 않았습니다.')

LS ELECTRIC DART 고유번호 조회 완료
2026년 1분기 DART 응답: 30 개 항목


,fs_div,account_nm,thstrm_nm,thstrm_amount,currency
0,CFS,유동자산,제 53 기1분기말,"3,734,613,316,299",KRW
1,CFS,비유동자산,제 53 기1분기말,"1,627,262,248,840",KRW
2,CFS,자산총계,제 53 기1분기말,"5,361,875,565,139",KRW
3,CFS,유동부채,제 53 기1분기말,"2,325,474,139,818",KRW
4,CFS,비유동부채,제 53 기1분기말,"902,232,390,462",KRW
5,CFS,부채총계,제 53 기1분기말,"3,227,706,530,280",KRW
6,CFS,자본금,제 53 기1분기말,"150,000,000,000",KRW
7,CFS,이익잉여금,제 53 기1분기말,"1,969,273,197,652",KRW
8,CFS,자본총계,제 53 기1분기말,"2,134,169,034,859",KRW
9,CFS,매출액,제 53 기1분기,"1,376,552,257,987",KRW


In [18]:
# 동일 기준(CFS 연결, 2026 1분기)만 비교. 여러 행이 있으면 임의의 첫 행을 채택하지 않음.
comparison=[]
for account, estimate in [('매출액',1376.6),('영업이익',126.6)]:
    x=dart_df.copy()
    if not x.empty and {'fs_div','account_nm','thstrm_amount'}.issubset(x.columns):
        x=x[(x['fs_div']=='CFS') & (x['account_nm'].astype(str).str.replace(' ','').isin([account, '수익(매출액)' if account=='매출액' else account]))]
    if len(x)==1:
        try:
            official=int(str(x.iloc[0]['thstrm_amount']).replace(',',''))/1e9
            comparison.append({'항목':account,'리포트_십억원':estimate,'DART_십억원':round(official,3),'차이_십억원':round(official-estimate,3),'상태':'수치 비교 수행(사후 공시; 차이는 기준 확인 필요)'})
        except (ValueError,TypeError): comparison.append({'항목':account,'리포트_십억원':estimate,'DART_십억원':None,'차이_십억원':None,'상태':'원본 금액 변환 불가'})
    else:
        comparison.append({'항목':account,'리포트_십억원':estimate,'DART_십억원':None,'차이_십억원':None,'상태':f'공식 행 {len(x)}건: 수동 확인 필요'})
result_df=pd.DataFrame(comparison)
display(result_df)
print('DART 전체 상태:',dart_status)

,항목,리포트_십억원,DART_십억원,차이_십억원,상태
0,매출액,1376.6,1376.552,-0.048,수치 비교 수행(사후 공시; 차이는 기준 확인 필요)
1,영업이익,126.6,126.567,-0.033,수치 비교 수행(사후 공시; 차이는 기준 확인 필요)


DART 전체 상태: 공식 API 응답 수신


## 5. 수주잔고 [도표 3] 원천 추적 및 미검증 범위

보고서 p.3은 **'전력인프라 수주잔고 추이'** 그래프에 `자료: Company Data, 교보증권 리서치센터`라고 표시합니다. 본문(p.1)에는 1Q26 전사 수주잔고 **5.6조원**이 기재되어 있지만, **전사 수주잔고와 도표의 전력인프라 수주잔고를 같은 값으로 간주하면 안 됩니다.** 그래프의 분기별 정확한 수치, 우측 축(%)이 의미하는 지표는 기업 IR 원문 확보 전까지 미검증입니다.

**추가로 해야 할 일:**
1. [LS ELECTRIC 공식 IR](https://www.ls-electric.com/ko/company/invest/ir/)에서 해당 발표 당시 1Q26 경영실적 PDF/Excel 다운로드
2. 분기별 '전력인프라 수주잔고'의 정의(전사/사업부, 연결/별도, 포함 계약 범위)를 확인
3. 도표 3의 분기별 수치 및 % 축의 정의를 IR과 비교하고, 원문 페이지를 기록

`IR 데이터 확인 완료` 같은 메시지는 원본 수치를 실제로 대조하기 전에는 표시하지 않습니다.

## 6. 희망 산출물: Power Research Agent

- **무엇을:** 증권사 리포트 속 숫자와 공시/IR 숫자의 차이를 확인하고 `일치 / 차이 있음 / 비교 불가`로 구분하는 검증형 Agent
- **누가:** 기업분석을 연습하는 스터디원
- **필요 데이터:** 리포트 PDF, DART 공식 API, LS ELECTRIC 분기 IR 자료
- **검증 규칙:** 분기·발표시점, 연결/별도, 잠정/확정/추정, 단위, 항목 정의 일치 여부
- **결과물:** 기업 브리프 + 근거 링크 + 오류/확인 필요 목록 + 실행로그

**현재 구현 수준:** PDF 텍스트 처리, 리포트 내부 수식 검산, 선택적 DART API 조회 및 비교표. LLM이 스스로 도구를 고르는 완전한 Agent나 수주잔고 데이터 자동추출은 **아직 구현되지 않았습니다**. 이는 추후 확장 목표입니다.

In [19]:
summary={
 '분석대상':'교보증권 LS ELECTRIC 2026-04-22',
 '리포트_PDF': str(pdf_path) if report_text else '미업로드',
 '내부_산술검증':'영업이익률·YoY 계산 통과',
 'DART_API': dart_status,
 '수주잔고_IR_교차검증':'미실시 — 공식 IR 원문 데이터 필요',
 '비고':'P(잠정), E(추정), 확정 공시의 발표시점 구분 필요',
}
print(pd.Series(summary).to_string())
# 비밀키나 공시 응답 전체는 저장하지 않습니다.
result_df.to_csv('w02_dart_comparison.csv',index=False,encoding='utf-8-sig')
print('표 저장: w02_dart_comparison.csv (실제 조회가 안 됐다면 미확인으로 표시)')

분석대상                교보증권 LS ELECTRIC 2026-04-22
리포트_PDF                     교보증권_LS일렉트릭 (1).pdf
내부_산술검증                         영업이익률·YoY 계산 통과
DART_API                           공식 API 응답 수신
수주잔고_IR_교차검증              미실시 — 공식 IR 원문 데이터 필요
비고              P(잠정), E(추정), 확정 공시의 발표시점 구분 필요
표 저장: w02_dart_comparison.csv (실제 조회가 안 됐다면 미확인으로 표시)


## 7. Gemini의 도구 선택을 포함한 제한형 에이전트 (선택 실행)

LLM에게 미리 허용한 **읽기 전용 도구** 중 다음 작업 하나를 고르게 합니다. `report_math`는 내부 계산 검토, `dart_summary`는 이미 수집한 DART 응답 상태 확인, `source_map`은 출처·미검증 항목 표시입니다. 새로운 외부 호출이나 송금/매매/파일 삭제는 허용하지 않습니다. 모델이 잘못된 JSON을 반환하거나 호출에 실패하면 안전하게 `source_map`으로 전환합니다.

이것은 **기본적인 LLM 도구 선택 1회**를 구현한 교육용 에이전트입니다. 완전 자율형 멀티스텝 연구 에이전트라고 주장하지 않습니다. `RUN_GEMINI_AGENT=True`로 바꿔야 실제 Gemini 호출이 발생합니다.

In [20]:
# Gemini를 이용한 제한형 도구 선택: 실행할 때만 True
RUN_GEMINI_AGENT = False
import json, re, importlib.util, subprocess, sys

def report_math():
    return {'1Q26_OPM_pct': round(126.6 / 1376.6 * 100, 3),
            '1Q26_sales_yoy_pct': round((1376.6 / 1032.1 - 1) * 100, 3),
            'source': '교보증권 p.2 도표 1 (리포트 내부 수치)'}

def dart_summary():
    return {'api_status': dart_status,
            'compared_rows': result_df.to_dict(orient='records'),
            'caution': 'DART 분기 공시와 2026-04-22 리포트 잠정치의 발표시점 차이 확인 필요'}

def source_map():
    return {'report': '교보증권 2026-04-22 p.2/3',
            'DART': 'https://opendart.fss.or.kr/',
            'IR': 'https://www.ls-electric.com/ko/company/invest/ir/',
            'unverified': 'p.3 전력인프라 수주잔고의 분기별 정확한 원자료 및 우측 축 정의'}

TOOLS = {'report_math': report_math, 'dart_summary': dart_summary, 'source_map': source_map}

def choose_tool_with_gemini(goal):
    if not GEMINI_KEY:
        return 'source_map', 'Gemini 키 미연결'
    try:
        from google import genai
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'google-genai'])
        from google import genai
    client = genai.Client(api_key=GEMINI_KEY)
    prompt = ('당신은 금융 출처 검증을 돕는 읽기 전용 에이전트다. '
              '사용자 요청에 맞는 도구 하나만 고르고 JSON 객체만 출력하라. '
              '허용 도구: report_math(리포트 내부 산술), dart_summary(DART 수집결과), '
              'source_map(출처와 미검증 항목). 답 형식: {"tool":"허용도구명"}. '
              '검증하지 않은 값을 사실처럼 말하지 마라. 요청: ' + goal)
    answer = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
    body = (answer.text or '').strip()
    match = re.search(r'\{[^{}]*\}', body)
    parsed = json.loads(match.group(0)) if match else {}
    tool = parsed.get('tool')
    return (tool, 'Gemini가 도구 선택') if tool in TOOLS else ('source_map', '허용되지 않은 선택 → 안전한 대체')

if RUN_GEMINI_AGENT:
    goal = 'LS ELECTRIC 리포트의 매출·영업이익 검증 상태를 점검해줘'
    try:
        tool_name, reason = choose_tool_with_gemini(goal)
    except Exception as exc:
        tool_name, reason = 'source_map', f'Gemini 호출 실패({type(exc).__name__}) → 안전한 대체'
    print('도구 선택:', tool_name, '|', reason)
    tool_result = TOOLS[tool_name]()
    print(json.dumps(tool_result, ensure_ascii=False, indent=2))
else:
    print('Gemini 호출 생략(RUN_GEMINI_AGENT=False). 다음은 로컬 검산 도구 실행:')
    print(json.dumps(report_math(), ensure_ascii=False, indent=2))

Gemini 호출 생략(RUN_GEMINI_AGENT=False). 다음은 로컬 검산 도구 실행:
{
  "1Q26_OPM_pct": 9.197,
  "1Q26_sales_yoy_pct": 33.379,
  "source": "교보증권 p.2 도표 1 (리포트 내부 수치)"
}


## 9. 최종 제출용 결과 보고서

In [21]:
# ── 최종 결과 출력: Colab의 맨 마지막에서 실행 ──
from IPython.display import display, Markdown
import pandas as pd

header = """# KFA AI Agent 스터디 | 2주차 사전과제

**대상:** LS ELECTRIC (010120)
**리포트:** 교보증권 김광식, 「1Q26 Review: 떠오르는 글로벌 배전 시장의 신성(新星)」
**발행일:** 2026.04.22 | **7쪽**

## 1. 선정 리포트
전력기기·전력인프라 사업의 분기 실적과 전력인프라 수주잔고를 함께 제시한다. 하나의 리포트에서 **공시로 확인 가능한 연결 재무정보**, **기업 IR이 필요한 사업별 실적 및 수주잔고**, **애널리스트 전망치**를 구분해 원천을 추적할 수 있다.

**[리포트 원문 링크](https://stock.pstatic.net/stock-research/company/62/20260422_company_926341000.pdf)** — PDF 파일 자체는 GitHub에 첨부하지 않음.

## 2. 선정 차트·표

| 선정 항목 | 리포트 위치 | 선정 이유 |
|---|---|---|
| 도표 1. LS ELECTRIC 실적 추이 및 전망 | 2쪽 | 매출액·영업이익의 확정/잠정/추정을 구별하고 공시·IR과 비교 가능 |
| 도표 3. 전력인프라 수주잔고 추이 | 3쪽 | 수주 확대라는 투자 논리의 핵심 지표로, 재무제표 API 외에 별도 IR 원천 필요 |

## 3. 데이터 소스 조사

| 수집 대상 | 원천 자료 | 기준·갱신 | 무료·수집 방법 |
|---|---|---|---|
| 연결 매출액·영업이익 | [OpenDART 재무정보](https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS003) | 2025년 분기·연간, 2026년 1분기 / 공시 주기 | 무료(인증키) / **① 공식 API** |
| 전력기기·전력인프라 별도 매출·이익 | [LS ELECTRIC 공식 IR](https://www.ls-electric.com/ko/company/invest/ir/) | 분기별 / IR 발표 시 | 무료 / **③ PDF 다운로드·대조** |
| 전력인프라 수주잔고 (도표 3) | [LS ELECTRIC IR](https://www.ls-electric.com/ko/company/invest/ir/) 및 사업보고서 | 1Q23~1Q26 / 분기 추적 | 무료 / **③ PDF·공시 조회**, 자동 추출 미검증 |
| 기업 연간 연결 재무정보 교차검증 | [LS ELECTRIC 경영성과](https://www.ls-electric.com/ko/company/invest/result) | 연간 / 결산 기준 | 무료 / **③ 웹·자료 다운로드** |

수집 방법 분류: ① 공식 API / ② 파이썬 라이브러리 / ③ 파일 다운로드 / ④ 크롤링 / ⑤ 자동 수집 불가.

## 4. 희망 산출물 — Power Research Agent

| 항목 | 계획 |
|---|---|
| 무엇을 | 증권사 보고서 속 실적·수주 지표를 추출하고 공시/IR 원문으로 출처 및 수치 성격(확정·잠정·추정)을 점검 |
| 누가 / 어떻게 | 기업분석 스터디원이 보고서 PDF와 종목 입력 → 출처 및 확인 필요 항목이 포함된 브리프 확인 |
| 필요 데이터 | DART 연결 재무정보, LS ELECTRIC 분기 IR, 수주잔고 자료, 해당 보고서의 표·차트 |
| 평가 기준 | 단위, 연결/별도, 분기, 기준일 일치 여부, 공식 출처 링크, 미확인 사항 명시 |
| 결과 형태 | 기업 브리프 5장 + 로그·출처 표 + 데이터 함정/미검증 목록 |

## 5. 추가 확인 및 한계

1. 도표 3의 정확한 분기별 수주잔고 원자료와 우측 축(%) 정의는 기업 IR에서 재확인 필요.
2. 리포트 1쪽의 '4분기' 서술과 제목·2쪽의 '1Q26 Review' 문맥 확인 필요.
3. 현재가 표기 기준일(01/28) 등 기준일 혼재로 해당 값을 본 과제 비교에 사용하지 않음.
4. 도표 1 FY25 매출액 **4,962.2십억원**과 기업 공식 실적 **4,965.799십억원**의 차이는 잠정치·집계 기준 확인 필요.
5. 과거 공시 수치도 잠정치 및 정정 여부를 확인해야 함.

---
## 6. 이번 노트북의 실제 실행 결과
"""
display(Markdown(header))

if 'report_data' in globals() and isinstance(report_data, pd.DataFrame):
    display(Markdown('### 6-1. 리포트 수치 (리포트 전사값 및 산식 재계산)'))
    display(report_data)
    display(Markdown(f"- **1Q26 영업이익률:** {126.6 / 1376.6 * 100:.2f}% (리포트 표기 9.2%)  \n- **1Q26 매출액 전년동기 성장률:** {(1376.6 / 1032.1 - 1)*100:.2f}% (리포트 표기 33.4%)  \n※ 이는 리포트 내부 수치의 **산술 검산**이며 공식 공시와의 일치 확인을 뜻하지 않음."))
else:
    display(Markdown('리포트 데이터 셀이 실행되지 않아 수치표를 표시할 수 없음.'))

display(Markdown('### 6-2. DART 공식 API 실행 및 비교 결과'))
display(Markdown('**DART 조회 상태:** ' + str(globals().get('dart_status', '실행 기록 없음'))))
if 'result_df' in globals() and isinstance(result_df, pd.DataFrame) and not result_df.empty:
    display(result_df)
else:
    display(Markdown('DART 비교 결과 없음 — 앞선 API·비교 셀의 실행 여부 확인 필요.'))

display(Markdown('### 6-3. 검증 범위 및 결론'))
display(Markdown("""- **검증 수행:** 보고서 전사값의 영업이익률·매출 성장률 계산, DART 연결 분기 재무정보 조회 및 조건부 비교.
- **DART 수치 대조의 한계:** 실제 API 응답의 행 수·보고서 종류·연결 기준·공시 시점이 맞아야 비교할 수 있음. 위 표의 상태를 확인해야 하며, 조회 성공 자체를 수치 일치로 해석하지 않음.
- **미완료:** 전력인프라 분기별 수주잔고의 IR 원본 추출 및 수치 전수 대조. 회사 전체 수주잔고 5.6조원을 전력인프라 부문 수주잔고로 간주하지 않음.
- **AI 기능:** Gemini 선택 실행 셀은 실제 활성화·실행 여부를 별도 확인해야 하며, 사용하지 않았다면 Agent 검증 완료로 표시하지 않음.

**정리:** 본 사전과제의 핵심은 원천 위치와 수집 경로 확인이다. 노트북에서 추가 실행한 API 검증은 그 결과와 한계를 별도 표시했다."""))


# KFA AI Agent 스터디 | 2주차 사전과제

**대상:** LS ELECTRIC (010120)  
**리포트:** 교보증권 김광식, 「1Q26 Review: 떠오르는 글로벌 배전 시장의 신성(新星)」  
**발행일:** 2026.04.22 | **7쪽**

## 1. 선정 리포트
전력기기·전력인프라 사업의 분기 실적과 전력인프라 수주잔고를 함께 제시한다. 하나의 리포트에서 **공시로 확인 가능한 연결 재무정보**, **기업 IR이 필요한 사업별 실적 및 수주잔고**, **애널리스트 전망치**를 구분해 원천을 추적할 수 있다.

**[리포트 원문 링크](https://stock.pstatic.net/stock-research/company/62/20260422_company_926341000.pdf)** — PDF 파일 자체는 GitHub에 첨부하지 않음.

## 2. 선정 차트·표

| 선정 항목 | 리포트 위치 | 선정 이유 |
|---|---|---|
| 도표 1. LS ELECTRIC 실적 추이 및 전망 | 2쪽 | 매출액·영업이익의 확정/잠정/추정을 구별하고 공시·IR과 비교 가능 |
| 도표 3. 전력인프라 수주잔고 추이 | 3쪽 | 수주 확대라는 투자 논리의 핵심 지표로, 재무제표 API 외에 별도 IR 원천 필요 |

## 3. 데이터 소스 조사

| 수집 대상 | 원천 자료 | 기준·갱신 | 무료·수집 방법 |
|---|---|---|---|
| 연결 매출액·영업이익 | [OpenDART 재무정보](https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS003) | 2025년 분기·연간, 2026년 1분기 / 공시 주기 | 무료(인증키) / **① 공식 API** |
| 전력기기·전력인프라 별도 매출·이익 | [LS ELECTRIC 공식 IR](https://www.ls-electric.com/ko/company/invest/ir/) | 분기별 / IR 발표 시 | 무료 / **③ PDF 다운로드·대조** |
| 전력인프라 수주잔고 (도표 3) | [LS ELECTRIC IR](https://www.ls-electric.com/ko/company/invest/ir/) 및 사업보고서 | 1Q23~1Q26 / 분기 추적 | 무료 / **③ PDF·공시 조회**, 자동 추출 미검증 |
| 기업 연간 연결 재무정보 교차검증 | [LS ELECTRIC 경영성과](https://www.ls-electric.com/ko/company/invest/result) | 연간 / 결산 기준 | 무료 / **③ 웹·자료 다운로드** |

수집 방법 분류: ① 공식 API / ② 파이썬 라이브러리 / ③ 파일 다운로드 / ④ 크롤링 / ⑤ 자동 수집 불가.

## 4. 희망 산출물 — Power Research Agent

| 항목 | 계획 |
|---|---|
| 무엇을 | 증권사 보고서 속 실적·수주 지표를 추출하고 공시/IR 원문으로 출처 및 수치 성격(확정·잠정·추정)을 점검 |
| 누가 / 어떻게 | 기업분석 스터디원이 보고서 PDF와 종목 입력 → 출처 및 확인 필요 항목이 포함된 브리프 확인 |
| 필요 데이터 | DART 연결 재무정보, LS ELECTRIC 분기 IR, 수주잔고 자료, 해당 보고서의 표·차트 |
| 평가 기준 | 단위, 연결/별도, 분기, 기준일 일치 여부, 공식 출처 링크, 미확인 사항 명시 |
| 결과 형태 | 기업 브리프 5장 + 로그·출처 표 + 데이터 함정/미검증 목록 |

## 5. 추가 확인 및 한계

1. 도표 3의 정확한 분기별 수주잔고 원자료와 우측 축(%) 정의는 기업 IR에서 재확인 필요.
2. 리포트 1쪽의 '4분기' 서술과 제목·2쪽의 '1Q26 Review' 문맥 확인 필요.
3. 현재가 표기 기준일(01/28) 등 기준일 혼재로 해당 값을 본 과제 비교에 사용하지 않음.
4. 도표 1 FY25 매출액 **4,962.2십억원**과 기업 공식 실적 **4,965.799십억원**의 차이는 잠정치·집계 기준 확인 필요.
5. 과거 공시 수치도 잠정치 및 정정 여부를 확인해야 함.

---
## 6. 이번 노트북의 실제 실행 결과


### 6-1. 리포트 수치 (리포트 전사값 및 산식 재계산)

,구간,상태,매출액_십억원,영업이익_십억원,영업이익률_계산(%)
0,1Q25,과거,1032.1,87.3,8.46
1,1Q26(P),잠정,1376.6,126.6,9.20
2,2Q26(E),추정,1536.7,156.7,10.20
3,FY25,잠정/기준확인,4962.2,426.9,8.60
4,FY26(E),추정,6281.7,671.2,10.69


- **1Q26 영업이익률:** 9.20% (리포트 표기 9.2%)  
- **1Q26 매출액 전년동기 성장률:** 33.38% (리포트 표기 33.4%)  
※ 이는 리포트 내부 수치의 **산술 검산**이며 공식 공시와의 일치 확인을 뜻하지 않음.

### 6-2. DART 공식 API 실행 및 비교 결과

**DART 조회 상태:** 공식 API 응답 수신

,항목,리포트_십억원,DART_십억원,차이_십억원,상태
0,매출액,1376.6,1376.552,-0.048,수치 비교 수행(사후 공시; 차이는 기준 확인 필요)
1,영업이익,126.6,126.567,-0.033,수치 비교 수행(사후 공시; 차이는 기준 확인 필요)


### 6-3. 검증 범위 및 결론

- **검증 수행:** 보고서 전사값의 영업이익률·매출 성장률 계산, DART 연결 분기 재무정보 조회 및 조건부 비교.
- **DART 수치 대조의 한계:** 실제 API 응답의 행 수·보고서 종류·연결 기준·공시 시점이 맞아야 비교할 수 있음. 위 표의 상태를 확인해야 하며, 조회 성공 자체를 수치 일치로 해석하지 않음.
- **미완료:** 전력인프라 분기별 수주잔고의 IR 원본 추출 및 수치 전수 대조. 회사 전체 수주잔고 5.6조원을 전력인프라 부문 수주잔고로 간주하지 않음.
- **AI 기능:** Gemini 선택 실행 셀은 실제 활성화·실행 여부를 별도 확인해야 하며, 사용하지 않았다면 Agent 검증 완료로 표시하지 않음.

**정리:** 본 사전과제의 핵심은 원천 위치와 수집 경로 확인이다. 노트북에서 추가 실행한 API 검증은 그 결과와 한계를 별도 표시했다.